## *RaschPy* simulation functionality

This notebook works through examples of how to generate simulated data sets with `RaschPy` for experimental use where knowledge of the underlying 'ground truth' of the generating parameters is useful, for example when comparing the efficacy of different estimation algorithms, such as in Elliott & Buttery (2022a) or exploring the effect of fitting different Rasch models to the same data set, such as in Elliott & Buttery (2022b). There are separate classes for each model: `SLM_Sim` for the simple logistic model (or dichotomous Rasch model) (Rasch, 1960), `PCM_Sim` for the partial credit model (Masters, 1982), `RSM_Sim` for the rating scale model (Andrich, 1978), `MFRM_Sim_Global` for the many-facet Rasch model (Linacre, 1994), and the family of extended MFRMs (Elliott 2025, Elliott & Buttery, 2022b): `MFRM_Sim_Items` for the vector-by-item extended MFRM, `MFRM_Sim_Thresholds` for the vector-by-threshold extended MFRM, `MFRM_Sim_Matrix` for the matrix extended MFRM, `MFRM_Sim_Bivector` for the bivector extended MFRM, and three closed-form stretch-restricted parameterisations: `MFRM_Sim_Centrality` (Jin & Wang 2018-style rater centrality/extremity, restricting `MFRM_Sim_Thresholds`), `MFRM_Sim_PseudoHalo` (an item-difficulty-compression model, restricting `MFRM_Sim_Items`), and `MFRM_Sim_Bistretch` (combining both stretch axes at once, restricting `MFRM_Sim_Bivector`). All data is generated to fit the chosen model.

**References**

&nbsp;&nbsp;&nbsp;&nbsp; Andrich, D. (1978). A rating formulation for ordered response categories. *Psychometrika*, *43*(4), 561–573.

&nbsp;&nbsp;&nbsp;&nbsp;   Elliott, M. (2025). Extended many-facet Rasch models: Accounting for rater effects in automated essay scoring systems [Apollo - University of Cambridge Repository]. https://doi.org/10.17863/CAM.127567

&nbsp;&nbsp;&nbsp;&nbsp; Elliott, M., & Buttery, P. J. (2022a) Non-iterative Conditional Pairwise Estimation for the Rating Scale Model, *Educational and Psychological Measurement*, *82*(5), 989-1019.

&nbsp;&nbsp;&nbsp;&nbsp; Elliott, M. and Buttery, P. J. (2022b) Extended Rater Representations in the Many-Facet Rasch Model, *Journal of Applied Measurement*, *22*(1), 133-160.

&nbsp;&nbsp;&nbsp;&nbsp; Jin, K.-Y., & Wang, W.-C. (2018). A new facets model for rater's centrality/extremity response style. *Journal of Educational Measurement*, *55*(4), 543–563.

&nbsp;&nbsp;&nbsp;&nbsp; Linacre, J. M. (1994). *Many-Facet Rasch Measurement*. MESA Press.

&nbsp;&nbsp;&nbsp;&nbsp; Masters, G. N. (1982). A Rasch model for partial credit scoring. *Psychometrika*, *47*(2), 149–174.

&nbsp;&nbsp;&nbsp;&nbsp; Rasch, G. (1960). *Probabilistic models for some intelligence and attainment tests*. Danmarks Pædagogiske
Institut.

Import the packages and set the working directory (here called `my_working_directory`) - you will save your output files here.

In [1]:
import raschpy as rp
import numpy as np
import pandas as pd
import os

os.chdir('my_working_directory')

### `MFRM_Sim_Bistretch`

Create an object `mfrm_sim_1` of the class `MFRM_Sim_Bistretch`. Unlike the fully-free `MFRM_Sim_Bivector`, each rater's (item x threshold) severity profile is generated from just three true parameters: a severity shift `lambda_r`, an item-difficulty-stretch `omega_items_r`, and a threshold-stretch `omega_thresholds_r` -- combining both stretch axes at once. Internally, `MFRM_Sim_Bistretch` first builds a preliminary `MFRM_Sim_Bivector` simulation to get realistic item locations, Rasch-Andrich thresholds and person locations -- so all the same `item_range`, `item_facet_range`, `threshold_facet_range`, `category_base`, `max_disorder`, `person_sd` and `offset` arguments apply here too, controlling that preliminary simulation (though `item_facet_range`/`threshold_facet_range` only affect the preliminary simulation's own free item/threshold effects, which are then entirely overwritten by the true 3-parameter stretch structure below -- `global_range` and `stretch_range` are what actually control the true generating parameters). `global_range` controls the spread of the true `lambda_r` values, and `stretch_range` controls the spread of both `omega_items_r` and `omega_thresholds_r` around 1 (use `manual_omega_items`/`manual_omega_thresholds` if you want them to differ). The implied item effect is `lambda_r + (omega_items_r - 1) * delta_i` (matching `MFRM_Sim_PseudoHalo`'s formula, carrying the rater's overall level); the implied threshold effect is `(omega_thresholds_r - 1) * tau_k` (matching `MFRM_Sim_Centrality`'s deviation term, but with no separate `lambda_r` here -- the bivector identification already puts the level entirely in the item axis). `lambda_r`/`omega_items_r`/`omega_thresholds_r` are sampled from a symmetric distribution by default; pass `manual_lambda`/`manual_omega_items`/`manual_omega_thresholds` to specify them directly instead. We pass `item_range=4`, `global_range=3` (spread of true `lambda_r`), `stretch_range=1` (spread of true `omega_items_r`/`omega_thresholds_r` around 1), `category_base=1.5` and `max_disorder=1`, plus `person_sd=2` and `offset=0.5`. One other required argument is `max_score`. There are 500 persons, 8 items and 10 raters, with no missing data for this simulation.

In [2]:
mfrm_sim_1 = rp.MFRM_Sim_Bistretch(no_of_items=8,
                                   no_of_persons=500,
                                   no_of_facet_elements=10,
                                   max_score=5,
                                   item_range=4,
                                   global_range=3,
                                   stretch_range=1,
                                   category_base=1.5,
                                   max_disorder=1,
                                   person_sd=2,
                                   offset=0.5,
                                   seed=42)

Save the generated response dataframe, which is stored as an attribute `mfrm_sim_1.responses`, to file, and view the first 5 lines.

In [3]:
mfrm_sim_1.responses.to_csv('mfrm_sim_1_responses.csv')
mfrm_sim_1.responses.head()

Item_1  Item_2  Item_3  Item_4  Item_5  Item_6  Item_7  \
Rater_1 Person_1       3       4       2       3       4       1       2   
        Person_2       3       2       2       1       2       0       1   
        Person_3       4       5       3       4       3       3       1   
        Person_4       3       5       3       5       5       2       2   
        Person_5       1       2       0       2       2       0       0   

                  Item_8  
Rater_1 Person_1       2  
        Person_2       2  
        Person_3       2  
        Person_4       2  
        Person_5       0

Save the generating item locations and Rasch-Andrich thresholds to file, and view them.

In [4]:
mfrm_sim_1.items.to_csv('mfrm_sim_1_items.csv', header=None)
mfrm_sim_1.items.head()

Item_1   -0.775991
Item_2   -1.774624
Item_3    0.267996
Item_4   -1.929639
Item_5   -0.809525
dtype: float64

In [5]:
mfrm_sim_1.thresholds.to_csv('mfrm_sim_1_thresholds.csv', header=None)
mfrm_sim_1.thresholds

1   -3.021099
2   -1.710138
3   -0.022473
4    1.682934
5    3.070775
dtype: float64

Like `MFRM_Sim_Bivector`, `MFRM_Sim_Bistretch` also stores the two implied rater-effect components separately: `mfrm_sim_1.item_effects` (a rater x item DataFrame, here implied by `lambda_r + (omega_items_r-1)*delta_i` rather than freely generated) and `mfrm_sim_1.threshold_effects` (a rater x threshold DataFrame, implied by `(omega_thresholds_r-1)*tau_k`). The reconstructed full rater x item x threshold matrix is also available as `mfrm_sim_1.facet_effects`, should you want it.

In [6]:
mfrm_sim_1.item_effects.to_csv('mfrm_sim_1_item_effects.csv')
mfrm_sim_1.item_effects.head()

,Item_1,Item_2,Item_3,Item_4,Item_5,Item_6,Item_7,Item_8
Rater_1,0.581508,0.747889,0.407571,0.773716,0.587095,0.135463,0.107281,0.277247
Rater_2,-0.782074,-1.221211,-0.322994,-1.289377,-0.796821,0.395192,0.469574,0.020976
Rater_3,0.693471,0.613196,0.777392,0.600735,0.690775,0.908678,0.922275,0.840270
Rater_4,0.011964,-0.260471,0.296772,-0.302760,0.002816,0.742326,0.788472,0.510167
Rater_5,-1.469359,-1.369098,-1.574173,-1.353535,-1.465992,-1.738145,-1.755127,-1.652706


In [7]:
mfrm_sim_1.threshold_effects.to_csv('mfrm_sim_1_threshold_effects.csv')
mfrm_sim_1.threshold_effects.head()

,1,2,3,4,5
Rater_1,-0.551208,-0.312020,-0.004100,0.307056,0.560272
Rater_2,0.429909,0.243356,0.003198,-0.239485,-0.436978
Rater_3,-1.479533,-0.837511,-0.011006,0.824189,1.503861
Rater_4,-1.019480,-0.577092,-0.007583,0.567912,1.036244
Rater_5,-0.610603,-0.345641,-0.004542,0.340143,0.620643


Unlike `MFRM_Sim_Bivector`, `MFRM_Sim_Bistretch` also exposes the true generating `lambda_r`/`omega_items_r`/`omega_thresholds_r` values directly (before they are expanded into `item_effects`/`threshold_effects` above) -- view and save them:

In [8]:
mfrm_sim_1.lambda_.to_csv('mfrm_sim_1_lambda.csv', header=None)
mfrm_sim_1.omega_items.to_csv('mfrm_sim_1_omega_items.csv', header=None)
mfrm_sim_1.omega_thresholds.to_csv('mfrm_sim_1_omega_thresholds.csv', header=None)
round(pd.DataFrame({'lambda': mfrm_sim_1.lambda_, 'omega_items': mfrm_sim_1.omega_items, 'omega_thresholds': mfrm_sim_1.omega_thresholds}), 3)

,lambda,omega_thresholds,omega_items
Rater_1,0.452221,1.182453,0.833391
Rater_2,-0.440842,0.857698,1.439738
Rater_3,0.755849,1.489733,1.080385
Rater_4,0.223661,1.337453,1.272808
Rater_5,-1.547267,1.202113,0.899602
Rater_6,1.452733,0.712451,0.687643
Rater_7,0.411750,0.945637,0.998624
Rater_8,0.491500,0.489733,0.439738
Rater_9,-1.387699,0.667182,1.279144
Rater_10,-0.411906,1.115547,1.068927


Save the generating person locations to file, and view the first 5 lines.

In [9]:
mfrm_sim_1.persons.to_csv('mfrm_sim_1_persons.csv', header=None)
mfrm_sim_1.persons.head()

Person_1    1.135686
Person_2   -1.553716
Person_3    2.027155
Person_4    2.407382
Person_5   -3.375818
dtype: float64

View `max_score`.

In [10]:
mfrm_sim_1.max_score

5

Create an object `mfrm_1` of the class `MFRM` from the response dataframe for analysis. The new object `mfrm_1` automatically inherits all the parameters from `mfrm_sim_1`, storing them under a namespace `.generating`.

In [11]:
mfrm_1 = rp.MFRM(mfrm_sim_1)

You may wish to create a simulation based on specified, known item locations, thresholds, person locations, and/or true `lambda_r`/`omega_items_r`/`omega_thresholds_r` values. This may be done by passing lists to the `manual_items`, `manual_thresholds`, `manual_persons`, `manual_lambda`, `manual_omega_items` and/or `manual_omega_thresholds` arguments (in which case there is no need to pass the relevant `item_range`, `item_facet_range`, `threshold_facet_range`, `category_base`, `max_disorder`, `person_sd`, `offset`, `global_range` or `stretch_range` arguments). You may also customise the names of the items and/or persons via `manual_item_names`/`manual_person_names`.

This is what is done in the example `mfrm_sim_2` below: a set of specified, fixed item locations (4 items with locations between -1.5 and +1.5 logits and maximum score of 5) and a set of Rasch-Andrich thresholds (summing to zero) are passed together with 5 raters' true `lambda_r`/`omega_items_r`/`omega_thresholds_r` values specified directly, and a random uniform distribution of person locations (between -2 and +2 logits). Raters 1-2 are neutral on all three parameters; Rater 3 has a mild lenient shift with a 'central' threshold-stretch and a halo-style item-stretch compression; Rater 4 has a stronger severity shift with the opposite item-stretch (exaggerating item-difficulty differences); Rater 5 combines a severity shift with a mild 'extreme' threshold-stretch and a near-neutral item-stretch. For this simulation, we also set a proportion of 10% missing data by passing `missing=0.1`.

In [12]:
mfrm_sim_2 = rp.MFRM_Sim_Bistretch(no_of_items=4,
                                   no_of_persons=500,
                                   no_of_facet_elements=5,
                                   max_score=5,
                                   missing=0.1,
                                   manual_persons=np.random.uniform(-2, 2, 500),
                                   manual_items=[-1.5, -0.5, 0.5, 1.5],
                                   manual_thresholds=[-2, -1, 0, 1, 2],
                                   manual_lambda=[0, 0, -0.5, 1, 0.5],
                                   manual_omega_items=[1, 1, 0.6, 1.5, 0.9],
                                   manual_omega_thresholds=[1, 1, 1.4, 1, 0.7],
                                   seed=42)

Save the generated response dataframe, which is stored as an attribute `mfrm_sim_2.responses`, to file, and view the first 5 lines.

In [13]:
mfrm_sim_2.responses.to_csv('mfrm_sim_2_responses.csv')
mfrm_sim_2.responses.head()

Item_1  Item_2  Item_3  Item_4
Rater_1 Person_1     3.0     3.0     1.0     NaN
        Person_2     5.0     1.0     1.0     0.0
        Person_3     5.0     3.0     2.0     0.0
        Person_4     NaN     2.0     2.0     1.0
        Person_5     4.0     5.0     2.0     1.0

View the item locations and Rasch-Andrich thresholds (as specified above).

In [14]:
mfrm_sim_2.items

Item_1   -1.5
Item_2   -0.5
Item_3    0.5
Item_4    1.5
dtype: float64

In [15]:
mfrm_sim_2.thresholds

1   -2
2   -1
3    0
4    1
5    2
dtype: int64

View the item effects and threshold effects (implied by the `lambda_`/`omega_items`/`omega_thresholds` values specified above).

In [16]:
mfrm_sim_2.item_effects

,Item_1,Item_2,Item_3,Item_4
Rater_1,0.00,0.00,0.00,0.00
Rater_2,0.00,0.00,0.00,0.00
Rater_3,0.10,-0.30,-0.70,-1.10
Rater_4,0.25,0.75,1.25,1.75
Rater_5,0.65,0.55,0.45,0.35


In [17]:
mfrm_sim_2.threshold_effects

,1,2,3,4,5
Rater_1,-0.0,-0.0,0.0,0.0,0.0
Rater_2,-0.0,-0.0,0.0,0.0,0.0
Rater_3,-0.8,-0.4,0.0,0.4,0.8
Rater_4,-0.0,-0.0,0.0,0.0,0.0
Rater_5,0.6,0.3,-0.0,-0.3,-0.6


View the true `lambda_r`/`omega_items_r`/`omega_thresholds_r` values specified above:

In [18]:
pd.DataFrame({'lambda': mfrm_sim_2.lambda_, 'omega_items': mfrm_sim_2.omega_items, 'omega_thresholds': mfrm_sim_2.omega_thresholds})

,lambda,omega_items,omega_thresholds
Rater_1,0.0,1.0,1.0
Rater_2,0.0,1.0,1.0
Rater_3,-0.5,0.6,1.4
Rater_4,1.0,1.5,1.0
Rater_5,0.5,0.9,0.7


View `max_score`.

In [19]:
mfrm_sim_2.max_score

5

Create an object, `mfrm_2`, of the class `MFRM` from the response dataframe for analysis.

In [20]:
mfrm_2 = rp.MFRM(mfrm_sim_2)

The two `MFRM` objects `mfrm_1` and `mfrm_2` are now available for analysis and, where appropriate, comparison of the recovered estimates with the generating estimates (including `mfrm_1.calibrate_bistretch()`'s `lambda_bistretch`/`omega_items_bistretch`/`omega_thresholds_bistretch` against `mfrm_sim_1.lambda_`/`mfrm_sim_1.omega_items`/`mfrm_sim_1.omega_thresholds`). See the example `Bistretch MFRM` notebook for details on how to run an `MFRM` analysis.